# Normalized & Imputate data

## Import Necessary library

In [8]:
import pandas as pd
from utils.preprocess.normalized.cpu import resolve_cpu_features
from utils.preprocess.normalized.mem_require import resolve_mem_require
from utils.preprocess.normalized.os import resolve_os_features
from utils.preprocess.normalized.price import resolve_price
from utils.preprocess.imputation.genres_tags.func import retrieve_tag_list
from utils.preprocess.normalized.release_date import resolve_release_date
from utils.preprocess.imputation.genres_tags.schemas import RetrieveRequest

### Load Data

In [ ]:
DATA_PATH = "data/steam-games/SteamGames_Clean_1.csv"
df = pd.read_csv(DATA_PATH)
df.head(5)

,Appid,Name,Type,ReleaseDate,Genres,Developers,Publishers,Description,price,Tags,ReviewScore,PositiveReview,NegativeReview,MemoryRequirement,Rank,CPU_Rank,CPU_Cores,OS_ID,OS_Tier
0,3764200,Resident Evil Requiem,game,"Feb 26, 2026","Action,Adventure","CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Resident Evil Requiem Deluxe Edition Resident ...,69.99$,"Gore,Violent,Action,Adventure,Horror,Survival ...",9,23715,793,16 GB RAM,1,622.0,6.0,1.0,4.0
1,730,Counter-Strike 2,game,"Aug 21, 2012","Action,Free To Play",Valve,Valve,"For over two decades, Counter-Strike has offer...",0$,"FPS,Shooter,Multiplayer,Competitive,Action,Tea...",8,1214897,202393,8 GB RAM,2,1991.0,4.0,2.0,3.0
2,1808500,ARC Raiders,game,"Oct 30, 2025",Action,Embark Studios,Embark Studios,ARC Raiders Deluxe Edition ARC Raiders Deluxe ...,39.99$,"Extraction Shooter,Multiplayer,PvP,PvE,Third-P...",8,149531,23194,12 GB RAM,3,1111.0,4.0,2.0,3.0
3,359550,Tom Clancy's Rainbow Six Siege,game,"Dec 1, 2015","Action,Free To Play",Ubisoft Montreal,Ubisoft,Season 11 Year 1 Overview Year 11 Roadmap Free...,0$,"Action,Multiplayer,Tactical,Shooter,Co-op,Sing...",8,522886,106839,8 GB RAM,4,661.0,4.0,2.0,3.0
4,1172470,Apex Legends™,game,"Nov 4, 2020","Action,Adventure,Free To Play",Respawn,Electronic Arts,Apex Legends: Breach About the Game Conquer wi...,0$,"Free to Play,Multiplayer,Battle Royale,FPS,Sho...",6,820,216,6 GB RAM,5,1704.0,2.0,2.0,3.0


## Normalized data
### MemoryRequirment
`MemoryRequirement` refers to minimum RAM to play the game. We need to remove the word **RAM** at the end of each word and normalized all unit back to GB


In [5]:
df['MemoryRequirement'] = df['MemoryRequirement'].apply(
    lambda mem : resolve_mem_require(mem) if pd.notna(mem) else pd.NA
).astype('Float64')

### OsRequirement

In [ ]:
os_nan = pd.Series([pd.NA, pd.NA], index=['OS_ID', 'OS_Tier'])

df[['OS_ID', 'OS_Tier']] = df['OsRequirement'].apply(
    lambda os : resolve_os_features(os) if pd.notna(os) else os_nan
)

### ReleaseDate

In [3]:
df['ReleaseDate'] = df['ReleaseDate'].apply(
    lambda date : resolve_release_date(date)
).astype('Int64')

### Price

In [9]:
df['price'] = df['price'].apply(
    lambda price : resolve_price(price)
).astype('Float64')

## Impuatation

### Impuate CpuRequirement

In [ ]:
core_nan = pd.Series([pd.NA, pd.NA], index=['CPU_Rank', 'CPU_Cores'])

df[['CPU_Rank', 'CPU_Cores']] = df['CpuRequirement'].apply(
    lambda cpu: resolve_cpu_features(cpu) if pd.notna(cpu) else core_nan
)

### Impuate Tags & Genres

In [ ]:
def impute_tags_for_row(row):
    # Keep existing tags if already present
    if pd.notna(row['Tags']):
        return row['Tags']

    # Handle potentially missing Genres or Description
    genres_list = row['Genres'].split(',') if pd.notna(row['Genres']) else []
    desc_text = str(row['Description']) if pd.notna(row['Description']) else ""

    response = retrieve_tag_list(RetrieveRequest(
        name=str(row['Name']),
        genres=genres_list,
        description=desc_text,
        top_k=10,
        top_tag=10,
        top_genre=3
    ))

    # Convert list of tags back into a comma-separated string for the CSV
    return ",".join(response.tag_list) if response.tag_list else None

def impute_genres_for_row(row):
    if pd.notna(row['Genres']):
        return row['Genres']

    game_name = row['Name'] if pd.notna(row['Name']) else ""
    desc_text = str(row['Description']) if pd.notna(row['Description']) else ""
    tags_list = row['Tags'].split(',') if pd.notna(row['Tags']) else []

    response = retrieve_tag_list(RetrieveRequest(
        name=game_name,
        description=desc_text,
        genres=tags_list,           # treat tags game as genres
        top_k=10,
        top_tag=10,
        top_genre=4,
    ))

    return ','.join(response.genres_list) if response.tag_list else None

# Apply across rows (axis=1 is required)
df['Tags'] = df.apply(impute_tags_for_row, axis=1)
df['Genres'] = df.apply(impute_genres_for_row, axis=1)

### Impuate OsRequirement
Using Tree-base prediction to predict `OsRequirement`, `OS_ID`, `OS_Tier`  
Predict `OS_ID` and label `OS_Tier` next

In [ ]:
import lightgbm as lbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, mean_squared_log_error


X = df[['ReleaseDate', 'price', 'CPU_Rank', 'CPU_Cores', 'OS_ID', 'OS_Tier', 'MemoryRequirement']]
X.dropna(inplace=True)
X.drop(columns=['OS_Tier'], inplace=True)   # Avoid leakage
y = X.pop('OS_ID')

X_train, X_test, y_train, y_test = train_test_split((X, y), test_size=0.25)

model = lbm.LGBMClassifier(
    learning_rate=0.1,
    max_depth=12,
    num_leaves=256,
    n_estimators=50
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print(f'Mean squared error: {f1_score(y_test, y_pred)}')

In [ ]:
# Predict `OS_ID` base on previous information
df['OS_ID'] = df.apply(
    lambda row : model.predict(row[['ReleaseDate', 'price', 'CPU_Rank', 'CPU_Cores', 'MemoryRequirement']])
    if pd.isna(row['OS_ID']) else row['OS_ID']
)

# Map `OS_ID` to `OS_Tier`
OS_ID_TIER_MAP = {
    1: 4, 2: 3, 3: 2, 4: 2,
    5: 1, 6: 1, 7: 1, 8: 1, 10: 1, 11: 1,
    42: 3, 30: 3
}

df['OS_Tier'] = df.apply(
    lambda row : OS_ID_TIER_MAP[row['OS_ID']]
    if pd.isna(row['OS_Tier']) else row['OS_Tier']
)

### Impuate OsRequirement
Using Tree-base prediction to predict `CpuRequirement`, `CPU_Rank`, `CPU_Cores`  
Predict `CPU_Rank` from that we can label `CPU_Cores`

In [ ]:
X = df[['ReleaseDate', 'price', 'CPU_Rank', 'CPU_Cores', 'OS_ID', 'OS_Tier', 'MemoryRequirement']]
X.dropna(inplace=True)
X.drop(columns=['CPU_Cores'], inplace=True)   # Avoid leakage
y = X.pop('CPU_Rank')

X_train, X_test, y_train, y_test = train_test_split((X, y), test_size=0.25)

model = lbm.LGBMClassifier(
    learning_rate=0.1,
    max_depth=12,
    num_leaves=256,
    n_estimators=50
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print(f'Mean squared error: {mean_squared_log_error(y_test, y_pred)}')

In [ ]:

df['CPU_Rank'] = df.apply(
    lambda row : model.predict(row[['ReleaseDate', 'price', 'OS_ID', 'OS_Tier', 'CPU_Cores', 'MemoryRequirement']])
    if pd.isna(row['CPU_Rank']) else row['CPU_Rank']
)

# TO DO NEXT: map `CPU_Rank` to `CPU_Cores`
# MAP from RANK to Cores( base one some simple statistical methods)
# NO need machine learning

### Impute MemoryRequirement

In [ ]:
X = df[['ReleaseDate', 'price', 'CPU_Rank', 'CPU_Cores', 'OS_ID', 'OS_Tier', 'MemoryRequirement']]
X.dropna(inplace=True)
y = X.pop('MemoryRequirement')

X_train, X_test, y_train, y_test = train_test_split((X, y), test_size=0.25)

model = lbm.LGBMClassifier(
    learning_rate=0.1,
    max_depth=12,
    num_leaves=256,
    n_estimators=50
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print(f'Mean squared error: {mean_squared_log_error(y_test, y_pred)}')

In [ ]:
df['MemoreRequirement'] = df.apply(
    lambda row : model.predict(row[['ReleaseDate', 'price', 'OS_ID', 'OS_Tier', 'CPU_Cores']])
    if pd.isna(row['MemoryRequirement']) else row['MemoryRequirement']
)

In [14]:
df.to_csv('data/SteamGames_Clean_2.csv', index=False)